# 22 — Only the missing40: 2B and4B LoRA-1 alone

Choose **A100 → Run all**, and allow Drive access.
Push the latest scripts, notebook, and `results/lora-missing40/` first.
No encoded bundle, new questions, training, limits, or free-control reruns.

**80 new answers total:** 2B on40, then4B on the same40. One attempt each.
These are exactly the frozen extra40. The existing424 LoRA-alone problems are excluded.
The completed2B LoRA + limit result remains **41.2% (191/464)**.

The2B original adapter must be on Drive:
`MyDrive/stop-overthinking/results/mini/lora/lora100/adapter_model.safetensors`.
The4B adapter is already in the project. Both hashes are checked before setup.

A LoRA is a trained add-on. Both runs use ordinary thinking, with **no thinking-token limit**.
Overall output cap:8,192 tokens. Seed3407.2B repeat stopping OFF;4B ON, matching earlier runs.

**Question:** how does LoRA alone compare with the free control across all464?
**Hypothesis:** completing the missing evidence may change the observed gap.
**Change:** apply the existing adapter with ordinary thinking.
**Measure:** passing code tests and output tokens.
**Compare:** saved2B limit1024 and4B limit2048 on identical IDs.
**Data:** the missing40 existing questions, not a new held-out test.
**Support/contradiction:** higher/lower LoRA accuracy, with problem-based uncertainty.

Default ceiling: **20 estimated compute units total for this notebook**, including setup and resumes.
This is a separate ledger; it does not reset Notebook21 or reuse its45-unit allowance.
Enter Colab's current units-per-hour rate and available balance.
The notebook saves a ZIP and releases the GPU at the end.
If setup stops early, disconnect manually; saved results are safe.


In [ ]:
UNIT_CAP = 20.0
RESERVE_UNITS = 1.0
GENERATION_RESERVE_UNITS = 2.0
RUN_READY = False
print("2B and4B LoRA-1 alone •40 each • no training •20-unit estimated ceiling")


## 1. GPU, Drive, GitHub, and frozen IDs

Check both adapters and completed-result IDs before package setup.
The exact extra40 contents and original424 source hashes must match.
No already completed LoRA-alone problem can enter this run.


In [ ]:
import os, sys, json, time, math, subprocess, hashlib, signal, shutil
from pathlib import Path
os.chdir("/content")
checked = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
if checked.returncode: raise RuntimeError("Choose A100 GPU first.")
GPU, MEMORY = [x.strip() for x in checked.stdout.strip().splitlines()[0].split(",")]
GPU_GB = float(MEMORY)/1024
if "A100" not in GPU or GPU_GB < 35: raise RuntimeError("Expected A100 with at least40GB.")
from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/stop-overthinking")
OUT = DRIVE / "results/lora-missing40"
OUT.mkdir(parents=True, exist_ok=True)
probe = OUT / ".write-check"; probe.write_text("ok"); probe.unlink()

def resource_number(prompt):
    value = float(input(prompt).strip())
    if not math.isfinite(value) or value <= 0: raise RuntimeError("Enter a positive Colab resource number.")
    return value
UNIT_RATE = resource_number("Compute units per hour (previous A100 rate6.77): ")
AVAILABLE_NOW = resource_number("Available compute units right now: ")
LEDGER = OUT / "compute-ledger.json"
if LEDGER.exists():
    budget = json.loads(LEDGER.read_text())
    BASE_USED = max(float(budget.get("estimated_units",0)), float(budget["initial_balance"])-AVAILABLE_NOW,0.0)
else:
    budget = {"version":2,"initial_balance":AVAILABLE_NOW,"sessions":0}
    BASE_USED = 0.0
SESSION_START = time.monotonic()
budget.update(estimated_units=BASE_USED,unit_rate=UNIT_RATE,unit_cap=UNIT_CAP,
    sessions=budget.get("sessions",0)+1,gpu=GPU,active=True)
def save_ledger(value):
    tmp = LEDGER.with_suffix(".tmp")
    with tmp.open("w") as f:
        json.dump(value,f,indent=2); f.flush(); os.fsync(f.fileno())
    os.replace(tmp,LEDGER)
def tick():
    budget["estimated_units"] = BASE_USED+(time.monotonic()-SESSION_START)/3600*UNIT_RATE
    save_ledger(budget)
def remaining_seconds(reserve=RESERVE_UNITS):
    tick()
    return max(0.0,(min(UNIT_CAP-budget["estimated_units"],AVAILABLE_NOW-(budget["estimated_units"]-BASE_USED))-reserve)/UNIT_RATE*3600)
save_ledger(budget)
if remaining_seconds()<60: raise RuntimeError("Budget too low; saved work remains safe.")
WORK = Path("/content/thesis")
REPO = "https://github.com/mahmudulhaquequdrati/stop-overthinking-thesis.git"
if not (WORK/".git").exists():
    subprocess.run(["git","clone","--depth","1","--branch","master",REPO,str(WORK)],check=True,timeout=300)
else:
    subprocess.run(["git","-C",str(WORK),"pull","--ff-only","origin","master"],check=True,timeout=300)
os.chdir(WORK); sys.path.insert(0,str(WORK/"scripts"))
MANIFEST = WORK/"results/lora-missing40/plan-manifest.json"
PROBLEMS = WORK/"results/extra/extra-lcb.json"
manifest = json.loads(MANIFEST.read_text())
for name,sha in manifest["script_hashes"].items():
    if hashlib.sha256((WORK/"scripts"/name).read_bytes()).hexdigest()!=sha: raise RuntimeError("Script revision mismatch: "+name)
from run_lora_missing40 import check_inputs
items = json.loads(PROBLEMS.read_text())["problems"]
assert len(items)==len({p["task_id"] for p in items})==40
manifest["problems_path"] = str(PROBLEMS)
check_inputs(WORK,manifest,{p["task_id"]:p for p in items})
ADAPTERS = {}
for model in ("2b","4b"):
    spec = manifest["adapters"][model]
    weights = DRIVE/"results/mini/lora/lora100/adapter_model.safetensors" if model=="2b" else WORK/"results/4b/raw/lora/lora1/adapter_model.safetensors"
    config = WORK/spec["config_path"]
    if not weights.exists(): raise RuntimeError("Missing adapter: "+str(weights))
    def sha_file(path):
        h=hashlib.sha256()
        with path.open("rb") as f:
            for part in iter(lambda:f.read(1024*1024),b""):h.update(part)
        return h.hexdigest()
    if sha_file(weights)!=spec["weight_sha256"] or sha_file(config)!=spec["config_sha256"]: raise RuntimeError("Adapter hash mismatch: "+model)
    dest=Path("/content/lora-missing40")/model
    dest.mkdir(parents=True,exist_ok=True)
    shutil.copy2(weights,dest/"adapter_model.safetensors")
    shutil.copy2(config,dest/"adapter_config.json")
    ADAPTERS[model]=dest
WHEELS = DRIVE/"wheels"; WHEELS.mkdir(parents=True,exist_ok=True)
print("Both adapters verified; exactly40 missing IDs per model. Existing424 excluded.")


## 2. Monitored commands and package setup

Reuse the successful Notebook21 setup. Logs and budget stay on Drive.


In [ ]:
def run_process(command, tag, timeout=1200, reserve=RESERVE_UNITS):
    allowed = min(timeout, remaining_seconds(reserve))
    if allowed < 30:
        print(f"Budget stop before {tag}; saved work is safe.")
        return False
    log = OUT / (tag + ".log")
    print("Starting", tag, "• maximum", round(allowed / 60, 1), "minutes", flush=True)
    with log.open("a", buffering=1) as stream:
        position = log.stat().st_size
        proc = subprocess.Popen(command, cwd=WORK, stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
        started = time.monotonic()
        try:
            while proc.poll() is None:
                tick()
                if time.monotonic() - started >= allowed or remaining_seconds(reserve) <= 0:
                    os.killpg(proc.pid, signal.SIGTERM)
                    try:
                        proc.wait(timeout=5)
                    except subprocess.TimeoutExpired:
                        os.killpg(proc.pid, signal.SIGKILL); proc.wait()
                    print(f"Stopped {tag} at the time/budget limit. Log: {log}")
                    return False
                with log.open() as reader:
                    reader.seek(position); tail = reader.read(); position = reader.tell()
                if tail:
                    print(tail[-3000:], end="", flush=True)
                time.sleep(3)
        except BaseException:
            if proc.poll() is None:
                os.killpg(proc.pid, signal.SIGTERM)
                try:
                    proc.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    os.killpg(proc.pid, signal.SIGKILL); proc.wait()
            raise
        finally:
            tick()
    if proc.returncode != 0:
        tail = log.read_text(errors="replace")[-5000:]
        raise RuntimeError(f"{tag} failed. Saved work is safe. Last log lines:\n{tail}")
    return True

def pip_install(*packages, tag="install-packages", timeout=1200):
    command = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *packages]
    if not run_process(command, tag, timeout):
        raise RuntimeError("Setup stopped at the time/budget limit. Read " + str(OUT / (tag + ".log")))


In [ ]:
pip_install("transformers>=5.15,<6", "peft>=0.20,<0.22", "accelerate", "safetensors",
            "datasets", "flash-linear-attention", "ninja", "packaging", "wheel")
# TorchAO is optional and unused by this 16-bit, unquantized inference run.
# Colab's old copy causes PEFT adapter injection to fail before generation.
if not run_process([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], "remove-unused-torchao", 120):
    raise RuntimeError("Not enough time to remove the incompatible optional TorchAO backend.")
print("Package installation finished. Model download has not started.")


In [ ]:
import urllib.request

def torch_info():
    probe = "import json,sys,torch; print(json.dumps(dict(ver=torch.__version__.split('+')[0], cuda=(torch.version.cuda or '').split('.')[0], abi=str(torch.compiled_with_cxx11_abi()).upper(), py='cp%d%d'%sys.version_info[:2])))"
    checked = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True, timeout=90)
    if checked.returncode:
        raise RuntimeError("PyTorch check failed: " + checked.stderr[-1500:])
    info = json.loads(checked.stdout.strip().splitlines()[-1])
    info["tag"] = ".".join(info["ver"].split(".")[:2])
    if not info["cuda"]:
        raise RuntimeError("PyTorch has no CUDA. Choose a fresh A100 runtime.")
    return info

def fast_check():
    probe = "import causal_conv1d, fla; from transformers.models.qwen3_5.modeling_qwen3_5 import Qwen3_5ForCausalLM"
    return subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True, timeout=120)

def wheel_tail(info):
    return f"+cu{info['cuda']}torch{info['tag']}cxx11abi{info['abi']}-{info['py']}-{info['py']}-linux_x86_64.whl"

request = urllib.request.Request("https://api.github.com/repos/Dao-AILab/causal-conv1d/releases/tags/v1.7.0", headers={"User-Agent": "thesis-notebook-22"})
assets = None
def released_wheel(info):
    global assets
    if assets is None:
        with urllib.request.urlopen(request, timeout=30) as response:
            assets = json.load(response).get("assets", [])
    return next((a for a in assets if wheel_tail(info) in a["name"]), None)

def install_ready(info):
    saved = sorted(p for p in WHEELS.glob("causal_conv1d*.whl") if wheel_tail(info) in p.name)
    if saved:
        try:
            pip_install("--no-deps", str(saved[0]), tag="install-convolution")
            if fast_check().returncode == 0:
                return True
        except RuntimeError as error:
            print("Saved wheel did not load:", str(error)[-500:])
    asset = released_wheel(info)
    if not asset:
        return False
    dest = WHEELS / asset["name"]
    temporary = str(dest) + ".part"
    download = "import urllib.request,os; urllib.request.urlretrieve(" + repr(asset["browser_download_url"]) + ", " + repr(temporary) + "); os.replace(" + repr(temporary) + ", " + repr(str(dest)) + ")"
    if not run_process([sys.executable, "-c", download], "download-wheel", 300):
        raise RuntimeError("Wheel download exceeded the remaining setup time.")
    pip_install("--no-deps", str(dest), tag="install-convolution")
    return fast_check().returncode == 0

info = torch_info()
print("Current environment:", info)
check = fast_check()
if check.returncode != 0 and not install_ready(info):
    print("No compatible ready wheel for this PyTorch/CUDA. Selecting a published ready-wheel stack.")
    target = dict(ver="2.9.1", tag="2.9", cuda="12", abi="TRUE", py=info["py"])
    if not released_wheel(target):
        raise RuntimeError("No published wheel for this Python version. Stop here; no source compilation is attempted.")
    pip_install("torch==2.9.1", "torchvision==0.24.1", "torchaudio==2.9.1",
                "--index-url", "https://download.pytorch.org/whl/cu128", tag="compatible-pytorch")
    info = torch_info()
    print("Compatible environment:", info)
    if not install_ready(info):
        raise RuntimeError("The compatible wheel did not load. See saved logs; model generation has not started.")
check = fast_check()
if check.returncode != 0:
    raise RuntimeError("FAST PATH OFF: " + check.stderr[-2000:])
print("FAST PATH ON")
run_process([sys.executable, "-m", "pip", "freeze"], "package-versions", 60)
os.chdir(WORK)


## 3. Check both adapters on CPU

Exercise PEFT injection before base-model downloads. No2B or4B inference yet.


In [ ]:
def worker(model,phase):
    folder=OUT/model; folder.mkdir(parents=True,exist_ok=True)
    return [sys.executable,str(WORK/"scripts/run_lora_missing40.py"),"--phase",phase,
        "--model",model,"--adapter",str(ADAPTERS[model]),"--problems",str(PROBLEMS),
        "--manifest",str(MANIFEST),"--root",str(WORK),"--out",str(folder),
        "--batch",str((64 if GPU_GB>=70 else 32) if model=="4b" else(128 if GPU_GB>=70 else 64))]
for model in ("2b","4b"):
    if not run_process(worker(model,"preflight"),"preflight-"+model,180):
        raise RuntimeError("CPU adapter check stopped: "+model)
RUN_READY=True
print("Both CPU compatibility checks passed.")


## 4. Run2B, then4B, grade, and report

Each model runs in a separate process, freeing GPU memory when it ends.
First4 answers measure execution speed and count toward40.
Saved answers and grades are skipped on resume. A partial run is labelled partial.
The results here are40-problem subset scores; the464 scores are recalculated later.


In [ ]:
from run_lora_missing40 import paths,read_answers
for model in ("2b","4b"):
    if not globals().get("RUN_READY",False):
        print("STOP: setup/CPU checks did not finish. No generation started.")
        break
    try:
        ok=run_process(worker(model,"generate"),"generate-"+model,24*3600,reserve=GENERATION_RESERVE_UNITS)
        if not ok:
            (OUT/model/"generation-status.json").write_text(json.dumps({"state":"budget_stopped"}))
        ans,_=paths(OUT/model)
        if read_answers(ans,True):
            run_process(worker(model,"grade"),"grade-"+model,24*3600,reserve=RESERVE_UNITS)
        else: print(model+": zero answers; skip grading. This is not0% accuracy.")
    except Exception as error:
        print(model+" FAILED:",str(error)[-4000:])
    try:
        run_process(worker(model,"report"),"report-"+model,120,reserve=0.1)
        report=OUT/model/"SUMMARY.md"
        if report.exists():
            from IPython.display import display,Markdown
            display(Markdown(report.read_text()))
    except Exception as error: print("Report unavailable:",str(error)[-2000:])
    if remaining_seconds(GENERATION_RESERVE_UNITS)<120:
        print("Budget stop. Reopen this notebook to resume saved work.")
        break
print("All available work saved:",OUT)


## 5. ZIP and disconnect

Download the ZIP and send it back for checked464-problem updates. The Drive copy stays saved.


In [ ]:
tick()
budget["active"] = False
save_ledger(budget)
ZIP_BASE = DRIVE / "results/lora-missing40-download"
try:
    shutil.make_archive(str(ZIP_BASE), "zip", root_dir=OUT)
    from google.colab import files
    print("Results ZIP:", str(ZIP_BASE) + ".zip")
    files.download(str(ZIP_BASE) + ".zip")
except Exception as error:
    print("Download was not started:", type(error).__name__, str(error)[:200])
    print("Your results are still saved on Drive:", OUT)
finally:
    print(f"Estimated charged time: {budget['estimated_units']:.2f}/{UNIT_CAP:.0f} units at the rate you entered.")
    print("Disconnecting this GPU runtime now. Reopen the same notebook to resume a partial run.")
    from google.colab import runtime
    runtime.unassign()
